In [ ]:
# =============================================================================
# CELL 1: DEPENDENCIES
# =============================================================================
# Install wheels from Kaggle datasets, fall back to PyPI for missing packages.
# No Betti build needed for submission.

import subprocess, sys, os, glob

# Try installing from local wheels first (offline Kaggle)
WHEEL_DIR = '/kaggle/input/datasets/sohier/vesuvius-metric-resources/wheels'
if os.path.isdir(WHEEL_DIR):
    wheels = glob.glob(os.path.join(WHEEL_DIR, '*.whl'))
    for whl in wheels:
        try:
            subprocess.check_call(
                [sys.executable, '-m', 'pip', 'install', whl, '-q', '--no-deps'],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
        except subprocess.CalledProcessError:
            pass  # Skip incompatible wheels silently
    print(f'Processed {len(wheels)} wheel files from {WHEEL_DIR}')
else:
    print(f'Wheel directory not found: {WHEEL_DIR}, using PyPI only')

# Install required packages (fallback to PyPI)
REQUIRED = ['tifffile', 'imagecodecs', 'scikit-image']
for pkg in REQUIRED:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f'Installed {pkg} from PyPI')

print('Dependencies ready')

In [ ]:
# =============================================================================
# CELL 2: IMPORTS + CONFIG + FULL MODEL ARCHITECTURE
# =============================================================================

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import time
import warnings
from pathlib import Path
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.ndimage import (
    binary_fill_holes, binary_closing, binary_opening,
    label, generate_binary_structure
)

warnings.filterwarnings('ignore')

# =============================================================================
# GPU INFO
# =============================================================================
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1e9
    print(f'GPU 0: {props.name} ({total_gb:.1f} GB)')

# =============================================================================
# CONFIGURATION
# =============================================================================
PATCH_SIZE = (192, 192, 192)
OVERLAP = 0.5
THRESHOLD = 0.50
MIN_COMPONENT_SIZE = 50
FEATURES = [32, 64, 128, 256, 320, 320]
N_BLOCKS = [1, 2, 3, 4, 6, 6]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Post-processing flags
PP_3D_CLOSING = True
PP_3D_HOLE_FILL = True
PP_SLICEWISE_CLOSE = True
PP_SLICEWISE_HOLE_FILL = True
PP_SLICEWISE_OPEN = True

# =============================================================================
# DATA PATHS — auto-detect test directory
# =============================================================================
TEST_DIR_CANDIDATES = [
    Path('/kaggle/input/vesuvius-challenge-surface-detection/test_images'),
    Path('/kaggle/input/vesuvius-challenge-2025/test'),
    Path('/kaggle/input/vesuvius-challenge-2025/test_images'),
    Path('/kaggle/input/vesuvius-challenge-surface-detection/test'),
]
TEST_DIR = None
for candidate in TEST_DIR_CANDIDATES:
    if candidate.exists():
        TEST_DIR = candidate
        break
if TEST_DIR is None:
    # Try to find any test directory under /kaggle/input
    for p in Path('/kaggle/input').rglob('test*'):
        if p.is_dir() and list(p.glob('*.tif')):
            TEST_DIR = p
            break

SAMPLE_SUB_CANDIDATES = [
    Path('/kaggle/input/vesuvius-challenge-surface-detection/sample_submission.csv'),
    Path('/kaggle/input/vesuvius-challenge-2025/sample_submission.csv'),
]
SAMPLE_SUB_PATH = None
for candidate in SAMPLE_SUB_CANDIDATES:
    if candidate.exists():
        SAMPLE_SUB_PATH = candidate
        break

print(f'Test directory: {TEST_DIR}')
print(f'Sample submission: {SAMPLE_SUB_PATH}')

# =============================================================================
# ALL 8 MODEL CHECKPOINT PATHS
# =============================================================================
FOLD0_BEST = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/best_model.pth')
FOLD0_CKPT_A = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_460.pth')
FOLD0_CKPT_B = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_440.pth')
FOLD0_CKPT_C = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/default/7/checkpoints_v11/fold_0/checkpoint_epoch_420.pth')

FOLD1_BEST = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/best_model.pth')
FOLD1_CKPT_A = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_100.pth')
FOLD1_CKPT_B = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_120.pth')
FOLD1_CKPT_C = Path('/kaggle/input/models/manish756/v11-vesuvius-model/pytorch/v11_fold1/1/checkpoints_v11/fold_1/checkpoint_epoch_110.pth')

ALL_CHECKPOINT_PATHS = [
    ('Fold0-Best', FOLD0_BEST),
    ('Fold0-Ep460', FOLD0_CKPT_A),
    ('Fold0-Ep440', FOLD0_CKPT_B),
    ('Fold0-Ep420', FOLD0_CKPT_C),
    ('Fold1-Best', FOLD1_BEST),
    ('Fold1-Ep100', FOLD1_CKPT_A),
    ('Fold1-Ep120', FOLD1_CKPT_B),
    ('Fold1-Ep110', FOLD1_CKPT_C),
]

# Check which checkpoints exist
print('\nCheckpoint availability:')
available_checkpoints = []
for name, path in ALL_CHECKPOINT_PATHS:
    exists = path.exists()
    status = 'FOUND' if exists else 'MISSING'
    print(f'  [{status}] {name}: {path}')
    if exists:
        available_checkpoints.append((name, path))

N_MODELS = len(available_checkpoints)
print(f'\nEnsemble size: {N_MODELS} models (SWA over predictions)')

# =============================================================================
# MODEL ARCHITECTURE — TopoPreservingUNet3D
# Identical to training. All 8 models use the same architecture.
# =============================================================================

def get_num_groups(channels, max_groups=32):
    """Find largest valid group count for GroupNorm."""
    for g in [max_groups, 16, 8, 4, 2, 1]:
        if channels % g == 0:
            return g
    return 1


class HybridConv3d(nn.Module):
    """Factored 3D conv: separate XY and Z convolutions."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid_ch = out_ch // 2
        self.conv_xy = nn.Conv3d(in_ch, mid_ch, kernel_size=(1, 3, 3), padding=(0, 1, 1), bias=False)
        self.conv_z = nn.Conv3d(in_ch, out_ch - mid_ch, kernel_size=(3, 1, 1), padding=(1, 0, 0), bias=False)
        self.norm = nn.GroupNorm(get_num_groups(out_ch), out_ch)
        self.act = nn.LeakyReLU(0.01, inplace=True)

    def forward(self, x):
        return self.act(self.norm(torch.cat([self.conv_xy(x), self.conv_z(x)], dim=1)))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_hybrid=False):
        super().__init__()
        if use_hybrid:
            self.conv = HybridConv3d(in_ch, out_ch)
        else:
            self.conv = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(get_num_groups(out_ch), out_ch),
                nn.LeakyReLU(0.01, inplace=True),
            )

    def forward(self, x):
        return self.conv(x)


class MultiScaleResBlock(nn.Module):
    def __init__(self, channels, scales=2, use_hybrid=False):
        super().__init__()
        self.scales = scales
        self.width = channels // scales
        self.convs = nn.ModuleList(
            [ConvBlock(self.width, self.width, use_hybrid=use_hybrid) for _ in range(scales - 1)]
        )
        self.norm = nn.GroupNorm(get_num_groups(channels), channels)

    def forward(self, x):
        splits = torch.chunk(x, self.scales, dim=1)
        outputs = [splits[0]]
        for i, conv in enumerate(self.convs):
            out = conv(splits[i + 1] + outputs[-1]) if i > 0 else conv(splits[i + 1])
            outputs.append(out)
        return x + self.norm(torch.cat(outputs, dim=1))


class AttentionBlock(nn.Module):
    """Channel + Spatial attention (CBAM-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(channels, max(channels // reduction, 8))
        self.fc2 = nn.Linear(max(channels // reduction, 8), channels)
        self.conv_spatial = nn.Conv3d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c = x.shape[:2]
        ca = torch.sigmoid(self.fc2(F.relu(self.fc1(self.gap(x).view(b, c))))).view(b, c, 1, 1, 1)
        x_ca = x * ca
        sa = torch.sigmoid(
            self.conv_spatial(
                torch.cat([x_ca.mean(1, keepdim=True), x_ca.max(1, keepdim=True)[0]], 1)
            )
        )
        return x_ca * sa


class SurfaceRefinementBlock(nn.Module):
    """
    Surface refinement with depth-tiling for T4 OOM safety.
    
    Conv3d at full 192^3 resolution needs massive im2col workspace:
      Conv3d(64, 32, 3x3x3) on (1, 64, 192, 192, 192) = 22.78 GiB
    TILE_DEPTH=32 reduces workspace to ~3.8 GiB, fitting in T4 15 GB.
    """
    TILE_DEPTH = 32

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.edge_conv = nn.Conv3d(in_ch, in_ch, 3, padding=1, bias=False)
        self.refine = nn.Sequential(
            nn.Conv3d(in_ch * 2, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(get_num_groups(out_ch), out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def _forward_tiled(self, x):
        """Process one sample via depth-wise tiling to avoid OOM."""
        B, C, D, H, W = x.shape
        assert B == 1
        tile = self.TILE_DEPTH
        pad = 1  # 3x3x3 conv boundary overlap

        output_slices = []
        for d_start in range(0, D, tile):
            d_end = min(d_start + tile, D)
            d_start_pad = max(0, d_start - pad)
            d_end_pad = min(D, d_end + pad)

            x_tile = x[:, :, d_start_pad:d_end_pad, :, :]
            edge_tile = torch.abs(self.edge_conv(x_tile))
            cat_tile = torch.cat([x_tile, edge_tile], dim=1)
            del x_tile, edge_tile

            out_tile = self.refine(cat_tile)
            del cat_tile

            valid_start = d_start - d_start_pad
            valid_end = valid_start + (d_end - d_start)
            output_slices.append(out_tile[:, :, valid_start:valid_end, :, :])
            del out_tile

        return torch.cat(output_slices, dim=2)

    def forward(self, x):
        B = x.shape[0]
        D = x.shape[2]
        # Small inputs: no tiling needed
        if D <= self.TILE_DEPTH:
            return self.refine(torch.cat([x, torch.abs(self.edge_conv(x))], dim=1))
        # Large inputs: tile each sample
        outputs = []
        for i in range(B):
            outputs.append(self._forward_tiled(x[i:i+1]))
        return torch.cat(outputs, dim=0)


class TopoPreservingUNet3D(nn.Module):
    """Topology-preserving 3D UNet with hybrid convolutions and attention."""
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_attention=True, use_hybrid_conv=True, use_surface_refinement=True,
                 use_deep_supervision=False):
        super().__init__()
        features = features or [32, 64, 128, 256, 320, 320]
        n_blocks = n_blocks or [1, 2, 3, 4, 6, 6]
        self.features = features
        self.use_deep_supervision = use_deep_supervision

        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()

        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            layers = [ConvBlock(in_channels, feat, use_hybrid=use_hybrid_conv and i > 0)]
            layers.extend([MultiScaleResBlock(feat, scales=2, use_hybrid=use_hybrid_conv) for _ in range(nb)])
            self.encoders.append(nn.Sequential(*layers))
            self.attentions.append(
                AttentionBlock(feat) if use_attention and i >= 2 else nn.Identity()
            )
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))

        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        for i in range(len(features) - 2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i + 1], features[i], 2, stride=2))
            if use_surface_refinement and i == 0:
                self.dec_convs.append(SurfaceRefinementBlock(features[i] * 2, features[i]))
            else:
                self.dec_convs.append(
                    nn.Sequential(
                        ConvBlock(features[i] * 2, features[i], use_hybrid=use_hybrid_conv),
                        MultiScaleResBlock(features[i], scales=2, use_hybrid=use_hybrid_conv),
                    )
                )
        self.final = nn.Conv3d(features[0], out_ch, 1)

    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = att(enc(x))
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        enc_features = enc_features[::-1]
        x = enc_features[0]
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.final(x)


# =============================================================================
# MODEL LOADING
# =============================================================================

def load_model_from_checkpoint(checkpoint_path, features=None, n_blocks=None):
    """
    Load a TopoPreservingUNet3D from checkpoint.
    Returns model in float32 on CPU.
    The caller is responsible for .to('cuda:0').half().
    """
    features = features or FEATURES
    n_blocks = n_blocks or N_BLOCKS

    model = TopoPreservingUNet3D(
        features=features,
        n_blocks=n_blocks,
        use_attention=True,
        use_hybrid_conv=True,
        use_surface_refinement=True,
        use_deep_supervision=False,
    )

    ckpt = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)
    state_dict = ckpt.get('model_state_dict', ckpt)
    # Handle torch.compile and DataParallel prefixes
    state_dict = {
        k.replace('_orig_mod.', '').replace('module.', ''): v
        for k, v in state_dict.items()
    }
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    epoch = ckpt.get('epoch', '?')
    dice = ckpt.get('best_dice', 0)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Loaded: {Path(checkpoint_path).name} (epoch={epoch}, dice={dice:.4f}, {n_params:.1f}M params)')

    del ckpt, state_dict
    gc.collect()
    return model


print('Architecture and model loader defined')
print(f'Config: patch={PATCH_SIZE}, overlap={OVERLAP}, threshold={THRESHOLD}')
print(f'Features: {FEATURES}')
print(f'N_blocks: {N_BLOCKS}')
print(f'SurfaceRefinementBlock: TILE_DEPTH={SurfaceRefinementBlock.TILE_DEPTH}')

In [ ]:
# =============================================================================
# CELL 3: INFERENCE + POST-PROCESSING + RLE FUNCTIONS
# =============================================================================

# ---- Normalization ----

def robust_zscore_normalize(img, lower_percentile=0.5, upper_percentile=99.5):
    """Robust Z-score normalization with percentile clipping (matches training)."""
    p_low = np.percentile(img, lower_percentile)
    p_high = np.percentile(img, upper_percentile)
    img_clipped = np.clip(img, p_low, p_high)
    mean = img_clipped.mean()
    std = img_clipped.std()
    return ((img_clipped - mean) / (std + 1e-8)).astype(np.float32)


# ---- Gaussian Weight ----

def create_gaussian_weight(patch_size, sigma=0.125):
    """Create 3D Gaussian weighting kernel for sliding window blending."""
    d, h, w = patch_size
    gz = np.exp(-0.5 * ((np.arange(d) - d / 2) / (d * sigma)) ** 2)
    gy = np.exp(-0.5 * ((np.arange(h) - h / 2) / (h * sigma)) ** 2)
    gx = np.exp(-0.5 * ((np.arange(w) - w / 2) / (w * sigma)) ** 2)
    return (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)


# ---- Patch Position Generation ----

def get_patch_positions(volume_shape, patch_size, overlap=0.5):
    """Generate all sliding window patch start positions."""
    D, H, W = volume_shape
    pd, ph, pw = patch_size
    sd = max(1, int(pd * (1 - overlap)))
    sh = max(1, int(ph * (1 - overlap)))
    sw = max(1, int(pw * (1 - overlap)))

    def _positions(total, patch, stride):
        if total <= patch:
            return [0]
        pos = list(range(0, total - patch + 1, stride))
        if (total - patch) not in pos:
            pos.append(total - patch)
        return pos

    z_pos = _positions(D, pd, sd)
    y_pos = _positions(H, ph, sh)
    x_pos = _positions(W, pw, sw)

    positions = []
    for z in z_pos:
        for y in y_pos:
            for x in x_pos:
                positions.append((z, y, x))
    return positions


# ---- Sliding Window Inference ----

@torch.no_grad()
def sliding_window_inference(model, volume_normalized, patch_size, overlap=0.5,
                             device='cuda', verbose=True):
    """
    Sliding window inference on an ALREADY-NORMALIZED volume.
    Returns probability map (after sigmoid), float32 numpy.
    Uses fp16 autocast for T4 memory efficiency.
    """
    model.eval()
    D, H, W = volume_normalized.shape
    pd, ph, pw = patch_size

    # Pad if volume is smaller than patch
    pad_d = max(0, pd - D)
    pad_h = max(0, ph - H)
    pad_w = max(0, pw - W)
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume_normalized = np.pad(
            volume_normalized, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect'
        )
        D, H, W = volume_normalized.shape
        if verbose:
            print(f'    Padded to: ({D}, {H}, {W})')

    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    weight_sum = np.zeros((D, H, W), dtype=np.float32)
    gauss = create_gaussian_weight(patch_size)

    positions = get_patch_positions((D, H, W), patch_size, overlap)
    if verbose:
        print(f'    Patches: {len(positions)}')

    pbar = tqdm(positions, desc='    Infer', leave=False) if verbose else positions
    for idx, (z, y, x) in enumerate(pbar):
        patch = volume_normalized[z:z + pd, y:y + ph, x:x + pw]

        # Safety pad if at edge and patch is smaller
        actual_shape = patch.shape
        if actual_shape != (pd, ph, pw):
            p = [
                (0, max(0, pd - actual_shape[0])),
                (0, max(0, ph - actual_shape[1])),
                (0, max(0, pw - actual_shape[2])),
            ]
            patch = np.pad(patch, p, mode='reflect')

        batch_tensor = torch.from_numpy(patch[None, None]).to(device).half()

        with torch.amp.autocast('cuda', dtype=torch.float16):
            pred = torch.sigmoid(model(batch_tensor))

        pred_np = pred[0, 0].float().cpu().numpy()

        pred_sum[z:z + pd, y:y + ph, x:x + pw] += pred_np * gauss
        weight_sum[z:z + pd, y:y + ph, x:x + pw] += gauss

        del batch_tensor, pred, pred_np, patch

        # Periodic cache clear to prevent fragmentation
        if idx % 20 == 0:
            torch.cuda.empty_cache()

    result = pred_sum / np.maximum(weight_sum, 1e-8)

    # Remove padding
    orig_D = D - pad_d if pad_d > 0 else D
    orig_H = H - pad_h if pad_h > 0 else H
    orig_W = W - pad_w if pad_w > 0 else W
    result = result[:orig_D, :orig_H, :orig_W]

    return result


# ---- Post-Processing Functions ----

def count_components(mask):
    """Count 26-connected components."""
    struct = generate_binary_structure(3, 3)  # 26-connectivity
    _, n = label(mask, structure=struct)
    return n


def remove_small_components(mask, min_size=50):
    """Remove connected components smaller than min_size."""
    struct = generate_binary_structure(3, 3)
    labeled, n = label(mask, structure=struct)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    small = sizes < min_size
    small[0] = False  # Background
    result = mask.copy()
    result[small[labeled]] = 0
    return result


def topology_safe_op(mask, operation_func, name='op'):
    """Apply operation only if it does not reduce component count (no merging)."""
    n_before = count_components(mask)
    result = operation_func(mask)
    n_after = count_components(result)
    if n_after < n_before:
        print(f'    [REVERT] {name}: would merge {n_before}->{n_after} components')
        return mask
    return result


def slicewise_hole_fill(mask):
    """Fill 2D holes in each slice across all 3 axes."""
    filled = mask.copy()
    for i in range(mask.shape[0]):
        filled[i] = binary_fill_holes(filled[i])
    for i in range(mask.shape[1]):
        filled[:, i, :] = binary_fill_holes(filled[:, i, :])
    for i in range(mask.shape[2]):
        filled[:, :, i] = binary_fill_holes(filled[:, :, i])
    return filled


def slicewise_morphology(mask, operation='close', iterations=1):
    """Apply morphological operation slice-by-slice across all 3 axes."""
    result = mask.copy()
    struct_2d = generate_binary_structure(2, 1)
    for axis in range(3):
        for i in range(mask.shape[axis]):
            if axis == 0:
                slc = result[i]
            elif axis == 1:
                slc = result[:, i, :]
            else:
                slc = result[:, :, i]
            if operation == 'close':
                slc_new = binary_closing(slc, structure=struct_2d, iterations=iterations)
            elif operation == 'open':
                slc_new = binary_opening(slc, structure=struct_2d, iterations=iterations)
            else:
                slc_new = slc
            if axis == 0:
                result[i] = slc_new
            elif axis == 1:
                result[:, i, :] = slc_new
            else:
                result[:, :, i] = slc_new
    return result


def postprocess(pred_prob, threshold=0.50, min_size=50,
                do_3d_closing=True, do_3d_hole_fill=True,
                do_slicewise_close=True, do_slicewise_hole_fill=True,
                do_slicewise_open=True, verbose=True):
    """
    Post-processing pipeline for ensemble predictions.
    
    Pipeline:
    1. Threshold
    2. Remove small components
    3. 3D closing (topology-safe)
    4. 3D hole fill (topology-safe)
    5. Slicewise closing (topology-safe)
    6. Slicewise hole fill (topology-safe)
    7. 2nd 3D hole fill (topology-safe)
    8. Slicewise opening (topology-safe)
    9. Final cleanup
    """
    if verbose:
        print('Post-processing:')
        print(f'  Input: min={pred_prob.min():.3f}, max={pred_prob.max():.3f}, mean={pred_prob.mean():.3f}')

    # 1. Threshold
    mask = (pred_prob > threshold).astype(np.uint8)
    if verbose:
        print(f'  1. Threshold ({threshold}): {mask.sum():,} voxels, FG={100 * mask.mean():.2f}%')

    if mask.sum() == 0:
        if verbose:
            print('  WARNING: Empty mask after threshold!')
        return mask

    # 2. Remove small components
    n_before = count_components(mask)
    mask = remove_small_components(mask, min_size)
    n_after = count_components(mask)
    if verbose:
        print(f'  2. Remove small (<{min_size}): {n_before}->{n_after} components')

    # 3. 3D binary closing (topology-safe)
    if do_3d_closing:
        struct_3d = generate_binary_structure(3, 3)  # 26-connectivity
        mask = topology_safe_op(
            mask,
            lambda m: binary_closing(m, structure=struct_3d, iterations=1).astype(np.uint8),
            '3D_closing'
        )
        if verbose:
            print(f'  3. 3D closing (26-conn): FG={100 * mask.mean():.2f}%')

    # 4. 3D hole filling (topology-safe)
    if do_3d_hole_fill:
        mask = topology_safe_op(
            mask,
            lambda m: binary_fill_holes(m).astype(np.uint8),
            '3D_fill_holes'
        )
        if verbose:
            print(f'  4. 3D hole fill: FG={100 * mask.mean():.2f}%')

    # 5. Slicewise closing (topology-safe)
    if do_slicewise_close:
        mask = topology_safe_op(
            mask,
            lambda m: slicewise_morphology(m, 'close', iterations=1),
            'slicewise_close'
        )
        if verbose:
            print(f'  5. Slicewise closing: FG={100 * mask.mean():.2f}%')

    # 6. Slicewise hole fill (topology-safe)
    if do_slicewise_hole_fill:
        mask = topology_safe_op(mask, slicewise_hole_fill, 'slicewise_hole_fill')
        if verbose:
            print(f'  6. Slicewise hole fill: FG={100 * mask.mean():.2f}%')

    # 7. Second 3D hole fill (catches holes created by slicewise ops)
    if do_3d_hole_fill:
        mask = topology_safe_op(
            mask,
            lambda m: binary_fill_holes(m).astype(np.uint8),
            '3D_fill_holes_2'
        )
        if verbose:
            print(f'  7. 3D hole fill (2nd): FG={100 * mask.mean():.2f}%')

    # 8. Slicewise opening (topology-safe)
    if do_slicewise_open:
        mask = topology_safe_op(
            mask,
            lambda m: slicewise_morphology(m, 'open', iterations=1),
            'slicewise_open'
        )
        if verbose:
            print(f'  8. Slicewise opening: FG={100 * mask.mean():.2f}%')

    # 9. Final cleanup
    mask = remove_small_components(mask, min_size)
    n_final = count_components(mask)
    if verbose:
        print(f'  Final: {n_final} components, {mask.sum():,} voxels, FG={100 * mask.mean():.2f}%')

    return mask


# ---- RLE Encoding ----

def rle_encode_3d(mask):
    """
    Run-length encode a 3D binary mask (Fortran/column-major order).
    Returns space-separated string of (start, length) pairs (1-indexed).
    Returns empty string for all-zero masks.
    """
    pixels = mask.flatten(order='F')
    if pixels.sum() == 0:
        return ''
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


print('Inference, post-processing, and RLE functions defined')

In [ ]:
# =============================================================================
# CELL 4: ENSEMBLE INFERENCE — PROCESS ALL TEST VOLUMES
# =============================================================================
#
# Memory strategy for T4 (15.6 GB VRAM) + 30 GB RAM:
# - Load ONE model at a time to GPU, run inference, accumulate prob, delete model
# - Never have more than 1 model + 1 volume in GPU memory
# - Model ~10M params x 2 bytes (fp16) = ~20 MB (tiny)
# - Volume inference working memory ~2-3 GB on GPU
# - Total GPU peak: ~3 GB per model pass -- fits easily in T4
# - prob_sum accumulates on CPU (numpy float32)
#
# Ensemble: Simple averaging (SWA) of all 8 model predictions.
# All models use identical architecture, so we just average probability maps.
# =============================================================================

def run_ensemble_inference():
    """Run 8-model SWA ensemble inference on all test volumes."""
    start_time = time.time()

    # ---- Discover test volumes ----
    test_volume_ids = []
    test_volume_paths = {}

    # Method 1: From sample_submission.csv
    if SAMPLE_SUB_PATH is not None and SAMPLE_SUB_PATH.exists():
        sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
        print(f'Sample submission columns: {list(sample_sub.columns)}')
        if 'id' in sample_sub.columns:
            test_volume_ids = list(sample_sub['id'].values)
            print(f'Found {len(test_volume_ids)} volume IDs from sample_submission.csv')

    # Method 2: From test directory
    if TEST_DIR is not None and TEST_DIR.exists():
        tif_files = sorted(TEST_DIR.glob('*.tif')) + sorted(TEST_DIR.glob('*.tiff'))
        for f in tif_files:
            vol_id = f.stem
            test_volume_paths[vol_id] = f
            if vol_id not in test_volume_ids:
                test_volume_ids.append(vol_id)
        print(f'Found {len(tif_files)} .tif files in {TEST_DIR}')

    # If volume IDs came from CSV but paths not found, search for them
    if TEST_DIR is not None:
        for vol_id in test_volume_ids:
            if vol_id not in test_volume_paths:
                for ext in ['.tif', '.tiff']:
                    candidate = TEST_DIR / f'{vol_id}{ext}'
                    if candidate.exists():
                        test_volume_paths[vol_id] = candidate
                        break
                # Also check subdirectories
                if vol_id not in test_volume_paths:
                    subdir = TEST_DIR / vol_id
                    if subdir.is_dir():
                        # Volume might be stored as directory of 2D slices
                        slices = sorted(subdir.glob('*.tif')) + sorted(subdir.glob('*.tiff'))
                        if slices:
                            test_volume_paths[vol_id] = subdir

    print(f'\nTotal volumes to process: {len(test_volume_ids)}')
    for vid in test_volume_ids:
        path = test_volume_paths.get(vid, 'NOT FOUND')
        print(f'  {vid}: {path}')

    if len(test_volume_ids) == 0:
        print('WARNING: No test volumes found! Creating empty submission.')
        return pd.DataFrame(columns=['id', 'rle'])

    # ---- Process each volume ----
    results = []

    for vol_idx, vol_id in enumerate(test_volume_ids):
        vol_start = time.time()
        print(f'\n{"=" * 70}')
        print(f'Volume [{vol_idx + 1}/{len(test_volume_ids)}]: {vol_id}')
        print(f'{"=" * 70}')

        # Clear GPU before starting
        torch.cuda.empty_cache()
        gc.collect()

        # Load volume
        vol_path = test_volume_paths.get(vol_id)
        if vol_path is None or not vol_path.exists():
            print(f'  ERROR: Volume file not found for {vol_id}, using empty mask')
            results.append({'id': vol_id, 'rle': ''})
            continue

        if vol_path.is_dir():
            # Load from directory of 2D slices
            slice_files = sorted(vol_path.glob('*.tif')) + sorted(vol_path.glob('*.tiff'))
            slices = [tifffile.imread(str(s)) for s in slice_files]
            volume_raw = np.stack(slices, axis=0).astype(np.float32)
            del slices
        else:
            volume_raw = tifffile.imread(str(vol_path)).astype(np.float32)

        original_shape = volume_raw.shape
        print(f'  Shape: {original_shape}')
        print(f'  Range: [{volume_raw.min():.1f}, {volume_raw.max():.1f}]')
        print(f'  Size: {volume_raw.nbytes / 1e9:.2f} GB')

        if volume_raw.ndim != 3:
            print(f'  ERROR: Expected 3D, got {volume_raw.ndim}D. Using empty mask.')
            results.append({'id': vol_id, 'rle': ''})
            del volume_raw
            gc.collect()
            continue

        # Normalize volume once (same normalization for all 8 models)
        print('  Normalizing...')
        volume_norm = robust_zscore_normalize(volume_raw)
        del volume_raw
        gc.collect()

        # Accumulator for ensemble averaging
        prob_sum = np.zeros(original_shape, dtype=np.float32)
        n_models_used = 0

        # ---- Run each model one at a time ----
        for model_idx, (model_name, ckpt_path) in enumerate(available_checkpoints):
            model_start = time.time()
            print(f'\n  Model [{model_idx + 1}/{N_MODELS}]: {model_name}')

            try:
                # Load model to CPU, then move to GPU in fp16
                model = load_model_from_checkpoint(ckpt_path)
                model = model.to(DEVICE).half()

                if torch.cuda.is_available():
                    alloc = torch.cuda.memory_allocated() / 1e9
                    print(f'    GPU mem after model load: {alloc:.2f} GB')

                # Run sliding window inference
                prob = sliding_window_inference(
                    model, volume_norm, PATCH_SIZE, OVERLAP,
                    device=DEVICE, verbose=True
                )
                print(f'    Prob: min={prob.min():.4f}, max={prob.max():.4f}, mean={prob.mean():.4f}')

                # Accumulate
                prob_sum += prob
                n_models_used += 1
                del prob

            except Exception as e:
                print(f'    ERROR: {type(e).__name__}: {e}')
                import traceback
                traceback.print_exc()

            finally:
                # Always clean up model from GPU
                try:
                    del model
                except NameError:
                    pass
                torch.cuda.empty_cache()
                gc.collect()

            model_elapsed = time.time() - model_start
            print(f'    Model time: {model_elapsed:.1f}s')

        # ---- Average and post-process ----
        if n_models_used == 0:
            print(f'  ERROR: No models succeeded for {vol_id}. Using empty mask.')
            results.append({'id': vol_id, 'rle': ''})
        else:
            print(f'\n  Averaging {n_models_used} model predictions...')
            avg_prob = prob_sum / n_models_used
            print(f'  Avg prob: min={avg_prob.min():.4f}, max={avg_prob.max():.4f}, mean={avg_prob.mean():.4f}')

            # Post-process
            mask = postprocess(
                avg_prob,
                threshold=THRESHOLD,
                min_size=MIN_COMPONENT_SIZE,
                do_3d_closing=PP_3D_CLOSING,
                do_3d_hole_fill=PP_3D_HOLE_FILL,
                do_slicewise_close=PP_SLICEWISE_CLOSE,
                do_slicewise_hole_fill=PP_SLICEWISE_HOLE_FILL,
                do_slicewise_open=PP_SLICEWISE_OPEN,
                verbose=True,
            )
            del avg_prob

            # RLE encode
            rle = rle_encode_3d(mask)
            results.append({'id': vol_id, 'rle': rle})
            rle_tokens = len(rle.split()) if rle else 0
            print(f'  RLE: {rle_tokens} tokens ({len(rle)} chars)')
            del mask

        # Clean up volume
        del prob_sum, volume_norm
        gc.collect()
        torch.cuda.empty_cache()

        vol_elapsed = time.time() - vol_start
        print(f'\n  Volume time: {vol_elapsed:.1f}s ({vol_elapsed / 60:.1f} min)')

    total_elapsed = time.time() - start_time
    print(f'\n{"=" * 70}')
    print(f'ENSEMBLE INFERENCE COMPLETE')
    print(f'  {len(results)} volumes, {N_MODELS} models, {total_elapsed:.1f}s ({total_elapsed / 60:.1f} min)')
    print(f'{"=" * 70}')

    return pd.DataFrame(results)


print('Ensemble inference function defined')
print(f'Will use {N_MODELS} models for SWA ensemble')

In [ ]:
# =============================================================================
# CELL 5: CREATE submission.csv
# =============================================================================

submission_df = run_ensemble_inference()
submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'\nSubmission saved: /kaggle/working/submission.csv')
print(f'Volumes: {len(submission_df)}')
print(submission_df.head())

In [ ]:
# =============================================================================
# CELL 6: VERIFY SUBMISSION
# =============================================================================

sub = pd.read_csv('/kaggle/working/submission.csv')
print(f'Volumes: {len(sub)}')
print(f'Columns: {list(sub.columns)}')
print(f'Any empty RLE: {sub["rle"].isna().sum()}')
print()
for _, row in sub.iterrows():
    rle_str = str(row['rle']) if pd.notna(row['rle']) else ''
    rle_tokens = len(rle_str.split()) if rle_str else 0
    print(f'  {row["id"]}: RLE tokens = {rle_tokens}, RLE chars = {len(rle_str)}')

print(f'\nSubmission file size: {Path("/kaggle/working/submission.csv").stat().st_size / 1e6:.2f} MB')
print('Submission verification complete.')